# 17 — Manager Analytics Q&A Smoke Test

این notebook آخرین backend اصلی پروژه را smoke-test می‌کند.

- KPIها deterministic هستند.
- LLM فقط متن مدیریتی می‌سازد.
- هر عدد باید از placeholder معتبر metric عبور کند.
- historical price و market-wide review-volume ranking غیرفعال‌اند.


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(
    PROJECT_ROOT
    / ".env"
)

from src.app.bootstrap import (
    create_app_services,
)

In [2]:
services = create_app_services(
    project_root=PROJECT_ROOT,
    api_key=os.environ[
        "METIS_API_KEY"
    ],
    base_url=os.environ[
        "METIS_BASE_URL"
    ],
)

analytics = (
    services
    .analytics
    .analytics
)

print(
    "Analytics products:",
    f"{len(analytics.repository.products):,}",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Analytics products: 948,352


In [3]:
category_table = (
    analytics.category_table(
        category_field="Category2",
        top_n=30,
        include_unknown=False,
    )
)

display(
    category_table.head(15)
)

SMOKE_CATEGORY = (
    category_table.iloc[0][
        "Category2"
    ]
)

print(
    "Smoke category:",
    SMOKE_CATEGORY,
)

,Category2,brand_count,product_count,review_count,products_with_reviews,review_coverage,median_price,avg_rating,rated_product_count,rated_product_coverage,weighted_product_rating,rating_count_total
0,دفتر,85,26458,82320,5928,0.224053,660000.0,85.978703,7137,0.269748,86.238371,213021
1,زیورآلات زنانه و مردانه,78,21925,59679,5208,0.237537,600000.0,79.721421,5406,0.246568,80.424989,129401
2,لباس دخترانه,279,21781,40525,5169,0.237317,1700000.0,78.508392,5779,0.265323,78.321594,65051
3,فکری و آموزشی,290,20612,215226,9499,0.460848,1250000.0,79.101839,9299,0.451145,80.734273,441073
4,زیورآلات زنانه,119,18045,51676,3697,0.204877,1000000.0,79.074535,5219,0.289221,79.635330,149143
5,تی شرت مردانه,187,18044,58911,5239,0.290346,2299000.0,70.421135,5110,0.283197,66.263000,162380
6,لباس پسرانه,243,17797,44046,4715,0.264932,1680000.0,78.066593,5406,0.303759,78.195269,65059
7,زیورآلات نقره زنانه,46,15872,12068,3042,0.191658,3257500.0,74.068687,3465,0.218309,74.170941,15245
8,گردنبند طلا زنانه,24,15743,0,0,0.000000,41200000.0,71.565079,630,0.040018,72.777828,4438
9,لباس زیر زنانه,188,12930,80764,5322,0.411601,1200000.0,77.520701,5362,0.414695,77.383518,222631


Smoke category: دفتر


In [4]:
filters = {
    "Category2": SMOKE_CATEGORY
}

overview = analytics.overview(
    filters=filters
)

print(
    "Product count:",
    overview[
        "product_count"
    ],
)

print(
    "Median price:",
    overview[
        "price"
    ][
        "median"
    ],
)

print(
    "Review coverage:",
    overview[
        "review_coverage"
    ],
)

print(
    "Rated product coverage:",
    overview[
        "product_rating"
    ][
        "rated_product_coverage"
    ],
)

print(
    "Weighted product rating /100:",
    overview[
        "weighted_product_rating"
    ],
)

Product count: 26458
Median price: 660000.0
Review coverage: 0.22405321641847456
Rated product coverage: 0.269748280293295
Weighted product rating /100: 86.23837086484431


In [5]:
QUESTION = (
    "وضعیت این دسته را از نظر اندازه کاتالوگ، "
    "قیمت، پوشش بازخورد و کیفیت امتیازدهی "
    "برای مدیر خلاصه کن."
)

result = (
    services
    .analytics
    .answer(
        question=QUESTION,
        filters=filters,
    )
)

print(
    "Numeric faithfulness:",
    result[
        "numeric_faithfulness_valid"
    ],
)

print(
    "Repaired:",
    result[
        "repaired"
    ],
)

print(
    "Confidence:",
    result[
        "confidence"
    ],
)

print()
print(
    result[
        "answer"
    ]
)

print()
print(
    "Validation errors:",
    result[
        "validation_errors"
    ],
)

Numeric faithfulness: True
Repaired: False
Confidence: medium

دسته «دفتر» از نظر کاتالوگ بزرگ است: 26,458 محصول با پوشش قیمت 100.0% ثبت شده است. میانه قیمت 660,000 تومان است و بخش میانی قیمت‌ها بین 480,000 تومان تا 888,000 تومان قرار دارد. پوشش بازخورد و امتیازدهی محدودتر است؛ تنها 22.4% از محصولات در کورپوس بررسی حضور دارند و 27.0% دارای امتیاز هستند. با وجود این محدودیت پوشش، کیفیت ادراک‌شده محصولات امتیازدار بالا است: میانگین وزنی امتیاز 4.31/5 است.

Validation errors: []


In [6]:
for insight in result[
    "insights"
]:
    print(
        "-",
        insight[
            "title"
        ],
    )
    print(
        " ",
        insight[
            "text"
        ],
    )

print()
print(
    "Caveats:",
    result[
        "caveats"
    ],
)

print()
print(
    "Telemetry:",
    result[
        "telemetry"
    ],
)

- اندازه و قیمت‌گذاری کاتالوگ
  کاتالوگ شامل 26,458 محصول است و قیمت برای 100.0% از محصولات در دسترس است؛ میانه قیمت 660,000 تومان گزارش می‌شود.
- شکاف پوشش بازخورد
  حضور بازخورد در کورپوس به 22.4% محدود است و پوشش محصول دارای امتیاز 27.0% است؛ بنابراین تعمیم کیفیت به کل کاتالوگ باید با احتیاط انجام شود.
- کیفیت امتیاز محصولات دارای داده
  محصولات دارای امتیاز، در مجموع 213,021 امتیاز ثبت‌شده دارند و میانگین وزنی امتیازشان 86.24/100 است.

Caveats: ['میانگین امتیاز فقط بر محصولات دارای تعداد امتیاز ثبت\u200cشده محاسبه شده است.', 'حضور بازخورد معتبر است، اما حجم بازخورد برای رتبه\u200cبندی فراگیر محبوبیت بازار استفاده نمی\u200cشود.', 'تحلیل برند کامل نیست، زیرا پوشش برندهای قابل شناسایی محدود است.', 'تحلیل قیمت تاریخی در این داده\u200cها در دسترس نیست.']

Telemetry: {'model': 'gpt-5.6-terra', 'generation_calls': 1, 'generation_latency_ms': 9421.231330999944, 'total_latency_ms': 10670.579351000924, 'prompt_tokens': 2564, 'completion_tokens': 696, 'total_tokens': 3260, 'estimated_cost_usd

In [7]:
compare_categories = (
    category_table[
        "Category2"
    ]
    .head(2)
    .tolist()
)

comparison = (
    analytics
    .compare_categories(
        compare_categories,
        category_field="Category2",
    )
)

display(
    comparison
)

compare_result = (
    services
    .analytics
    .answer(
        question=(
            "این دو دسته را از نظر قیمت، پوشش بازخورد "
            "و تعداد امتیاز ثبت‌شده مقایسه کن."
        ),
        comparison_categories=(
            compare_categories
        ),
        category_field=(
            "Category2"
        ),
    )
)

print(
    "Comparison numeric faithfulness:",
    compare_result[
        "numeric_faithfulness_valid"
    ],
)

print(
    compare_result[
        "answer"
    ]
)

,Category2,product_count,brand_count,review_count,review_coverage,median_price,price_coverage,avg_product_rating,weighted_product_rating,rating_count_total,weighted_review_rating
0,دفتر,26458,85,82320,0.224053,660000.0,0.999698,85.978703,86.238371,213021,4.310039
1,زیورآلات زنانه و مردانه,21925,78,59679,0.237537,600000.0,1.000000,79.721421,80.424989,129401,4.000983


Comparison numeric faithfulness: True
در مقایسه دو دسته، میانه قیمت «دفتر» 660,000 تومان و بالاتر از «زیورآلات زنانه و مردانه» با 600,000 تومان است. پوشش بازخورد در زیورآلات 23.8% و اندکی بالاتر از دفتر با 22.4% است. اما دفتر از نظر تعداد امتیاز ثبت‌شده با 213,021 در برابر 129,401 جلوتر است.
